**Requests** -Downloading static HTML pages  
**Beautiful Soup**-Parsing HTML and extracting data   
**lxml-Fast HTML** parsing for Beautiful Soup  
**Playwright**-Scraping JavaScript-rendered pages  
**Scrapy**-Crawling many pages with a structured framework  
**ScrapingBee**-Handling JavaScript rendering, proxies, screenshots, anti-bot issues, and structured extraction through an API  
**pandas**-Cleaning, inspecting, and exporting scraped data

Check whether the site has an API.  
If the data is in the raw HTML, use Requests and Beautiful Soup.  
If JavaScript renders the data, use Playwright.  
If you need to crawl many pages, use Scrapy.  

In [5]:
!pip install requests beautifulsoup4 lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 173.8 kB/s  0:00:301.4 kB/s eta 0:00:02


In [7]:
from __future__ import annotations

import re
from dataclasses import dataclass
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from requests import RequestException


print("imports handled")

imports handled


In [9]:
HN_URL = "https://news.ycombinator.com/"
TIMEOUT_SECONDS = 10

HEADERS = {
    "User-Agent": "python-web-scraping-tutorial/1.0"
}


@dataclass
class Story:
    rank: int
    title: str
    url: str
    points: int
    author: str
    age: str
    comments: int


def extract_number(text: str | None) -> int:
    """Return the first integer in a string, or 0 if none exists."""
    if not text:
        return 0

    match = re.search(r"\d+", text.replace("\xa0", " "))
    return int(match.group()) if match else 0


def fetch_html(url: str) -> str:
    """Download a page and return its HTML."""
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=TIMEOUT_SECONDS,
        )
        response.raise_for_status()
    except RequestException as exc:
        raise RuntimeError(f"Failed to fetch {url}: {exc}") from exc

    return response.text


def parse_hacker_news(html: str, limit: int = 5) -> list[Story]:
    """Extract Hacker News stories from HTML."""
    soup = BeautifulSoup(html, "lxml")
    stories: list[Story] = []

    for story_row in soup.select("tr.athing.submission")[:limit]:
        title_link = story_row.select_one(".titleline > a")

        if title_link is None:
            continue

        href = title_link.get("href")
        story_url = urljoin(HN_URL, href if isinstance(href, str) else "")

        subtext_row = story_row.find_next_sibling("tr")
        subtext = subtext_row.select_one(".subtext") if subtext_row else None

        rank_text = story_row.select_one(".rank")
        score_text = subtext.select_one(".score") if subtext else None
        author_link = subtext.select_one(".hnuser") if subtext else None
        age_link = subtext.select_one(".age a") if subtext else None

        comment_link = None
        if subtext:
            for link in subtext.select("a"):
                link_text = link.get_text(" ", strip=True).lower()
                if "comment" in link_text or link_text == "discuss":
                    comment_link = link
                    break

        stories.append(
            Story(
                rank=extract_number(rank_text.get_text() if rank_text else None),
                title=title_link.get_text(" ", strip=True),
                url=story_url,
                points=extract_number(score_text.get_text() if score_text else None),
                author=author_link.get_text(" ", strip=True) if author_link else "unknown",
                age=age_link.get_text(" ", strip=True) if age_link else "unknown",
                comments=extract_number(comment_link.get_text() if comment_link else None),
            )
        )

    return stories


html = fetch_html(HN_URL)
stories = parse_hacker_news(html)

for story in stories:
    print(f"{story.rank}. {story.title}")
    print(f"   URL: {story.url}")
    print(f"   {story.points} points by {story.author} | {story.age} | {story.comments} comments")
    print()

1. AI agent bankrupted their operator while trying to scan DN42
   URL: https://lantian.pub/en/article/fun/ai-agent-bankrupted-their-operator-scan-dn42lantian.lantian/
   630 points by xiaoyu2006 | 5 hours ago | 236 comments

2. If you are asking for human attention, demonstrate human effort
   URL: https://tombedor.dev/human-attention-and-human-effort/
   782 points by jjfoooo4 | 11 hours ago | 259 comments

3. Nobody ever gets credit for fixing problems that never happened (2001) [pdf]
   URL: https://web.mit.edu/nelsonr/www/Repenning=Sterman_CMR_su01_.pdf
   481 points by sam_bristow | 10 hours ago | 163 comments

4. Show HN: Homebrew 6.0.0
   URL: https://brew.sh/2026/06/11/homebrew-6.0.0/
   1247 points by mikemcquaid | 18 hours ago | 299 comments

5. Report on an Unidentified Space Station
   URL: https://sseh.uchicago.edu/doc/roauss.htm
   31 points by paulmooreparks | 3 hours ago | 6 comments

